In [ ]:
import os
import re
import pandas as pd
import nltk
import pdfplumber

In [ ]:
# Define the folder containing meeting minutes
TXT_FOLDER = "/Users/emilymoore/Downloads/DS 4002 Project 1/meeting_mins"

# Define a list of uncertainty-related keywords
# These capture forward-looking language, risk, ambiguity, or policy hesitation
UNCERTAINTY_WORDS = [
    "uncertain","uncertainty","risk","risks","may","might",
    "could","potential","possibly","unknown","unclear",
    "concern","concerns","challenge","challenges",
    "pending","depend","depends"
]


# Define housing-related keywords
# These ensure we focus specifically on housing-policy discussions
HOUSING_WORDS = [
    "housing","affordable housing","zoning",
    "development","residential","rent",
    "apartment","units","subsidy"
]

# Function to extract full text from a .txt file
def extract_text_from_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

# Function to count occurrences of words in a text
def count_words(text, word_list):
    lower_text = text.lower() # Convert text to lowercase for case-insensitive matching
    return sum(lower_text.count(word.lower()) for word in word_list) # Count how many times each keyword appears and sum them

# Function to extract meeting start and end times using regex
def get_meeting_times(text):
    # Regex pattern to capture meeting start time
    # Added \s* everywhere to be safe with line breaks in text files
    start_pattern = r"calls\s+meeting\s+to\s+order\s*(?:at)?\s*((\d{1,2}:\d{2})\s*([AaPp][Mm])?)"
    end_pattern = r"adjourned\s*(?:at)?\s*((\d{1,2}:\d{2})\s*([AaPp][Mm])?)"
    # Search for patterns in the full text
    start_match = re.search(start_pattern, text, re.IGNORECASE)
    end_match = re.search(end_pattern, text, re.IGNORECASE)
    # Return extracted times if found; otherwise None
    return {
        "start_time": start_match.group(1).strip() if start_match else None,
        "end_time": end_match.group(1).strip() if end_match else None
    }

results = [] # Initialize an empty list to store results for each meeting file

# Loop through all files in the meeting minutes folder
for filename in os.listdir(TXT_FOLDER):
    if filename.endswith(".txt"): # Process only .txt files
        path = os.path.join(TXT_FOLDER, filename)
        
        full_text = extract_text_from_txt(path) # Extract full text from the file
        times = get_meeting_times(full_text) # Extract meeting start and end times

        housing_mentions = count_words(full_text, HOUSING_WORDS) # Count total housing-related keyword mentions
        uncertainty_mentions = count_words(full_text, UNCERTAINTY_WORDS) # Count total uncertainty-related keyword mentions

        sentences = nltk.sent_tokenize(full_text.lower()) # Tokenize text into sentences using NLTK
        # Convert to lowercase to standardize matching

        # Identify sentences that contain BOTH a housing-related word AND an uncertainty-related word
        # This captures "housing uncertainty density"
        housing_uncertainty_sentences = [
            s for s in sentences
            if any(h in s for h in HOUSING_WORDS)
            and any(u in s for u in UNCERTAINTY_WORDS)
        ]

        # Append structured results for this file
        results.append({
            "file_name": filename,
            "start_time": times["start_time"],
            "end_time": times["end_time"],
            "housing_mentions": housing_mentions,
            "uncertainty_mentions": uncertainty_mentions,
            "housing_uncertainty_sentences": len(housing_uncertainty_sentences)
        })

df_mins = pd.DataFrame(results) # Convert list of dictionaries into a DataFrame
print(df_mins) # Print the structured output to verify extraction worked

# Save the results
df_mins.to_csv(os.path.join(TXT_FOLDER, "housing_uncertainty_results.csv"), index=False)